# fhir4ds.cql

Clinical Quality Language (CQL) to SQL translation. This notebook demonstrates how CQL expressions are translated into highly optimized DuckDB SQL for population-scale analytics.

In [ ]:
# Install the unified package with measure evaluation support
%pip install "fhir4ds-v2[measures]"

In [ ]:
import json
import fhir4ds
from fhir4ds.cql import CQLToSQLTranslator, FHIRDataLoader
from fhir4ds.cql.parser import parse_cql

# Create connection and register UDFs
connection = fhir4ds.create_connection()

# Load sample data using FHIRDataLoader (creates the proper schema)
loader = FHIRDataLoader(connection)
resources = [
    {"resourceType": "Patient", "id": "p1", "active": True, "birthDate": "1990-06-15"},
    {"resourceType": "Patient", "id": "p2", "active": False, "birthDate": "1985-04-22"},
    {"resourceType": "Encounter", "id": "e1", "status": "finished",
     "subject": {"reference": "Patient/p1"}},
]
for r in resources:
    loader.load_resource(r)

### Parse and Translate
Translate CQL logic into inspectable SQL fragments. FHIR4DS supports clinical functions like `AgeInYears()` natively in the database.

In [ ]:
cql_text = """
library Test version '1.0'
using FHIR version '4.0.1'
context Patient

define "Has Encounter":
    exists ([Encounter] E where E.status = 'finished')

define "Status":
    if Patient.active then 'Active' else 'Inactive'
"""

library = parse_cql(cql_text)
translator = CQLToSQLTranslator(connection=connection)
definitions = translator.translate_library(library)

print(f"SQL for 'Has Encounter':\n{definitions['Has Encounter'].to_sql()}")

### Population SQL
The `translate_library_to_population_sql` method generates a single query that evaluates all definitions for the entire patient population in one scan.

In [ ]:
pop_sql = translator.translate_library_to_population_sql(
    library,
    output_columns={
        "has_encounter": "Has Encounter",
        "label": "Status"
    }
)
connection.execute(pop_sql).df()